# 01 Data Preparation

**Purpose.** This notebook prepares the county-year modeling dataset for soybean yield forecasting.

It combines:

1. Google Earth Engine exports containing county-month satellite, weather, and soil-moisture features.
2. USDA county-level soybean outcomes containing yield, harvested acres, and production.

The final output is a county-year model-ready table saved as:

`data/processed/model_ready_soybean_15states_2010_2025.csv`

This file is used by `02_yield_prediction_models.ipynb`.


In [1]:
# ============================================================
# 0. Setup
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount("/content/gdrive")

PROJECT_DIR = Path("/content/gdrive/MyDrive/Soybean_Project_GEE")

# Input folders
GEE_DIR = PROJECT_DIR
USDA_DIR = PROJECT_DIR / "USDA_County_Outcomes"

# Output folder
PROCESSED_DIR = PROJECT_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


## 1. Read Google Earth Engine Feature Exports


The satellite and environmental features were extracted using Google Earth Engine in separate state-year batches to avoid oversized export tasks. Each exported CSV contains county-month observations for 15 major U.S. soybean-producing states from 2010 to 2025. These states account for approximately 90% of U.S. soybean production in the sample and include Arkansas, Iowa, Illinois, Indiana, Kansas, Kentucky, Michigan, Minnesota, Missouri, Mississippi, North Dakota, Nebraska, Ohio, South Dakota, and Wisconsin. The extraction covers May through September because this period spans the main U.S. soybean growing season, including planting, vegetative growth, reproductive development, and seed filling. These stages are when satellite greenness, rainfall, heat stress, and soil-moisture conditions are most informative for final yield.

The extraction covers May through September, which corresponds to the main U.S. soybean growing season. This window captures planting and early vegetative growth in late spring, mid-season canopy development, and the critical reproductive and seed-filling stages during summer and early fall. These months are therefore most relevant for measuring crop greenness, rainfall, heat stress, and soil-moisture conditions that affect final soybean yield.

The feature extraction uses a soybean cropland mask so that county-level averages are computed primarily over soybean-growing pixels rather than the full county area. This helps align the remote-sensing variables with soybean production conditions instead of broader land-cover conditions.

The extracted features include:

* `ndvi`: Normalized Difference Vegetation Index, used as a proxy for vegetation greenness and crop vigor.
* `evi`: Enhanced Vegetation Index, which is less prone to saturation than NDVI in dense vegetation and can better capture crop canopy conditions.
* `rain_mm`: Monthly accumulated precipitation, capturing water availability during the growing season.
* `heat_days_gt35c`: Number of days with daily maximum temperature above 35°C, used to measure extreme heat stress.
* `soil_moisture_l2`: Soil-moisture indicator, used to capture water availability and drought-related growing conditions.

These variables were selected because soybean yield is affected by crop greenness, canopy development, rainfall, heat stress, and soil-water availability during the growing season. The monthly structure allows the model to compare early-season and late-season forecasting windows, such as July-end, August-end, and September-end signals.


In [2]:
# ============================================================
# 1. Read Google Earth Engine output files
# ============================================================

gee_files = [
    GEE_DIR / "Soybean_Features_B1_IL_IA_IN_MN_NE_2010_2025_May_Sep_Soil_V1.csv",
    GEE_DIR / "Soybean_Features_B2_MO_OH_ND_SD_AR_2010_2025_May_Sep_Soil_V1.csv",
    GEE_DIR / "Soybean_Features_B3_KS_MS_MI_WI_KY_2010_2025_May_Sep_Soil_V1.csv",
]

for f in gee_files:
    if not f.exists():
        raise FileNotFoundError(f"Missing GEE export file: {f}")

gee_df = pd.concat(
    [pd.read_csv(f) for f in gee_files],
    ignore_index=True
)

print("GEE raw rows:", len(gee_df))
print("GEE raw columns:", gee_df.columns.tolist())
display(gee_df.head())


GEE raw rows: 106560
GEE raw columns: ['state_alpha', 'state_name', 'county_name', 'county_ansi', 'geoid', 'year', 'month', 'ndvi', 'evi', 'rain_mm', 'heat_days_gt35c', 'soil_moisture_l2']


,state_alpha,state_name,county_name,county_ansi,geoid,year,month,ndvi,evi,rain_mm,heat_days_gt35c,soil_moisture_l2
0,IL,ILLINOIS,Marion,121,17121,2010,5,0.445579,0.283119,93.202729,0.0,0.461480
1,IL,ILLINOIS,Bond,5,17005,2010,5,0.407684,0.250075,148.856827,0.0,0.461938
2,IL,ILLINOIS,Jersey,83,17083,2010,5,0.392097,0.245199,97.321419,0.0,0.376680
3,IL,ILLINOIS,St. Clair,163,17163,2010,5,0.417601,0.267519,143.508176,0.0,0.409668
4,IL,ILLINOIS,Clinton,27,17027,2010,5,0.431403,0.268802,142.635223,0.0,0.462833


## 2. Read USDA county outcomes

`county_outcomes` is the target table. It was previously downloaded from the USDA API and saved as a CSV.  
For this GitHub version, the notebook reads the saved result directly instead of calling the API every time.

The table should contain at least:

- `state_alpha`
- `county_ansi`
- `county_name`
- `year`
- `yield_bu_acre`
- `harvested_acres`
- `production`


In [3]:
# ============================================================
# 2. Read USDA county-level soybean outcomes
# ============================================================

# This CSV was created from the USDA API in an upstream extraction step.
# We read the saved file here to keep this notebook fast and reproducible.

county_outcomes_path = USDA_DIR / "Soybean_Real_County_Outcomes_15CandidateStates_2010_2025.csv"

if not county_outcomes_path.exists():
    raise FileNotFoundError(f"Missing USDA outcome file: {county_outcomes_path}")

county_outcomes = pd.read_csv(county_outcomes_path)

print("USDA outcome rows:", len(county_outcomes))
print("USDA outcome columns:", county_outcomes.columns.tolist())
display(county_outcomes.head())


USDA outcome rows: 14600
USDA outcome columns: ['state_alpha', 'county_ansi', 'county_name', 'year', 'yield_bu_acre', 'harvested_acres', 'production_bu']


,state_alpha,county_ansi,county_name,year,yield_bu_acre,harvested_acres,production_bu
0,IL,11,BUREAU,2010,55.0,119600,6582000
1,IL,15,CARROLL,2010,58.9,34000,2004000
2,IL,73,HENRY,2010,56.6,151300,8561000
3,IL,85,JO DAVIESS,2010,55.1,32600,1796000
4,IL,103,LEE,2010,56.2,91600,5152000


## 3. Clean identifiers and numeric columns

County codes must be standardized before merging.  
The merge key is:

`state_alpha + county_ansi + year`

This is necessary because different states can have the same county code.


In [4]:
# ============================================================
# 3A. Clean GEE feature table
# ============================================================

gee_keep_cols = [
    "state_alpha",
    "state_name",
    "county_ansi",
    "county_name",
    "geoid",
    "year",
    "month",
    "ndvi",
    "evi",
    "rain_mm",
    "heat_days_gt35c",
    "soil_moisture_l2",
]

missing_gee_cols = [c for c in gee_keep_cols if c not in gee_df.columns]
if missing_gee_cols:
    raise ValueError(f"Missing GEE columns: {missing_gee_cols}")

gee_clean = gee_df[gee_keep_cols].copy()

gee_clean["state_alpha"] = (
    gee_clean["state_alpha"]
    .astype("string")
    .str.strip()
    .str.upper()
)

gee_clean["county_name"] = (
    gee_clean["county_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

gee_clean["county_ansi"] = (
    pd.to_numeric(gee_clean["county_ansi"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(3)
)

gee_clean["year"] = pd.to_numeric(gee_clean["year"], errors="coerce").astype("Int64")
gee_clean["month"] = pd.to_numeric(gee_clean["month"], errors="coerce").astype("Int64")

for col in ["ndvi", "evi", "rain_mm", "heat_days_gt35c", "soil_moisture_l2"]:
    gee_clean[col] = pd.to_numeric(gee_clean[col], errors="coerce")

gee_clean = gee_clean.dropna(subset=["state_alpha", "county_ansi", "year", "month"]).copy()
gee_clean["year"] = gee_clean["year"].astype(int)
gee_clean["month"] = gee_clean["month"].astype(int)

print("GEE clean rows:", len(gee_clean))
print("Years:", gee_clean["year"].min(), "-", gee_clean["year"].max())
print("Months:", sorted(gee_clean["month"].unique()))
print("States:", sorted(gee_clean["state_alpha"].unique()))

display(gee_clean.head())


GEE clean rows: 106560
Years: 2010 - 2025
Months: [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
States: ['AR', 'IA', 'IL', 'IN', 'KS', 'KY', 'MI', 'MN', 'MO', 'MS', 'ND', 'NE', 'OH', 'SD', 'WI']


,state_alpha,state_name,county_ansi,county_name,geoid,year,month,ndvi,evi,rain_mm,heat_days_gt35c,soil_moisture_l2
0,IL,ILLINOIS,121,MARION,17121,2010,5,0.445579,0.283119,93.202729,0.0,0.461480
1,IL,ILLINOIS,005,BOND,17005,2010,5,0.407684,0.250075,148.856827,0.0,0.461938
2,IL,ILLINOIS,083,JERSEY,17083,2010,5,0.392097,0.245199,97.321419,0.0,0.376680
3,IL,ILLINOIS,163,ST. CLAIR,17163,2010,5,0.417601,0.267519,143.508176,0.0,0.409668
4,IL,ILLINOIS,027,CLINTON,17027,2010,5,0.431403,0.268802,142.635223,0.0,0.462833


In [6]:
county_outcomes.head()

,state_alpha,county_ansi,county_name,year,yield_bu_acre,harvested_acres,production_bu
0,IL,11,BUREAU,2010,55.0,119600,6582000
1,IL,15,CARROLL,2010,58.9,34000,2004000
2,IL,73,HENRY,2010,56.6,151300,8561000
3,IL,85,JO DAVIESS,2010,55.1,32600,1796000
4,IL,103,LEE,2010,56.2,91600,5152000


In [8]:
# ============================================================
# 3B. Clean USDA outcome table
# ============================================================

required_outcome_cols = [
    "state_alpha",
    "county_ansi",
    "county_name",
    "year",
    "yield_bu_acre",
    "harvested_acres",
    "production_bu",
]

missing_outcome_cols = [c for c in required_outcome_cols if c not in county_outcomes.columns]
if missing_outcome_cols:
    raise ValueError(f"Missing USDA outcome columns: {missing_outcome_cols}")

county_outcomes = county_outcomes.copy()

county_outcomes["state_alpha"] = (
    county_outcomes["state_alpha"]
    .astype("string")
    .str.strip()
    .str.upper()
)

county_outcomes["county_name"] = (
    county_outcomes["county_name"]
    .astype("string")
    .str.strip()
    .str.upper()
)

county_outcomes["county_ansi"] = (
    pd.to_numeric(county_outcomes["county_ansi"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(3)
)

county_outcomes["year"] = pd.to_numeric(county_outcomes["year"], errors="coerce").astype("Int64")

for col in ["yield_bu_acre", "harvested_acres", "production_bu"]:
    county_outcomes[col] = pd.to_numeric(county_outcomes[col], errors="coerce")

county_outcomes = county_outcomes.dropna(
    subset=["state_alpha", "county_ansi", "year", "yield_bu_acre"]
).copy()

county_outcomes["year"] = county_outcomes["year"].astype(int)

print("USDA clean rows:", len(county_outcomes))
print("Years:", county_outcomes["year"].min(), "-", county_outcomes["year"].max())
print("States:", sorted(county_outcomes["state_alpha"].unique()))

display(county_outcomes.head())


USDA clean rows: 14600
Years: 2010 - 2025
States: ['AR', 'IA', 'IL', 'IN', 'KS', 'KY', 'MI', 'MN', 'MO', 'MS', 'ND', 'NE', 'OH', 'SD', 'WI']


,state_alpha,county_ansi,county_name,year,yield_bu_acre,harvested_acres,production_bu
0,IL,011,BUREAU,2010,55.0,119600,6582000
1,IL,015,CARROLL,2010,58.9,34000,2004000
2,IL,073,HENRY,2010,56.6,151300,8561000
3,IL,085,JO DAVIESS,2010,55.1,32600,1796000
4,IL,103,LEE,2010,56.2,91600,5152000


## 4. Pivot monthly GEE features to county-year format

The GEE table is originally county-month level.  
For modeling, each county-year should have one row, with monthly features expanded into columns:

- `ndvi_05`, `ndvi_06`, ..., `ndvi_09`
- `evi_05`, `evi_06`, ..., `evi_09`
- and so on.


In [9]:
# ============================================================
# 4. Pivot monthly GEE features to county-year format
# ============================================================

feature_cols = [
    "ndvi",
    "evi",
    "rain_mm",
    "heat_days_gt35c",
    "soil_moisture_l2",
]

pivot_keys = [
    "state_alpha",
    "county_ansi",
    "year",
]

# Check uniqueness before pivoting
duplicated = gee_clean.duplicated(
    subset=pivot_keys + ["month"],
    keep=False
)

print("Duplicated county-year-month rows:", duplicated.sum())

if duplicated.any():
    display(
        gee_clean.loc[duplicated]
        .sort_values(pivot_keys + ["month"])
        .head(20)
    )
    raise ValueError("Resolve duplicated county-year-month rows before pivoting.")

gee_wide = (
    gee_clean
    .pivot(
        index=pivot_keys,
        columns="month",
        values=feature_cols
    )
)

gee_wide.columns = [
    f"{feature}_{int(month):02d}"
    for feature, month in gee_wide.columns
]

gee_wide = gee_wide.reset_index()

print("GEE county-year rows after pivot:", len(gee_wide))
display(gee_wide.head())


Duplicated county-year-month rows: 0
GEE county-year rows after pivot: 21312


,state_alpha,county_ansi,year,ndvi_05,ndvi_06,ndvi_07,ndvi_08,ndvi_09,evi_05,evi_06,evi_07,evi_08,evi_09,rain_mm_05,rain_mm_06,rain_mm_07,rain_mm_08,rain_mm_09,heat_days_gt35c_05,heat_days_gt35c_06,heat_days_gt35c_07,heat_days_gt35c_08,heat_days_gt35c_09,soil_moisture_l2_05,soil_moisture_l2_06,soil_moisture_l2_07,soil_moisture_l2_08,soil_moisture_l2_09
0,AR,001,2010,0.378456,0.633280,0.819685,0.743902,0.429014,0.250452,0.520525,0.684343,0.571511,0.282429,101.508432,26.202610,57.109210,77.154271,41.600745,0.000000,12.976556,8.929358,17.443118,6.199984,0.410144,0.323402,0.262680,0.231959,0.254878
1,AR,001,2011,0.315863,0.525159,0.792616,0.823436,0.548853,0.221749,0.408518,0.654815,0.679658,0.358816,148.903966,49.979712,55.333011,125.499785,52.947497,4.365595,28.071577,28.244094,23.833896,6.000000,0.400547,0.282187,0.250128,0.254487,0.269607
2,AR,001,2012,0.414466,0.710187,0.833227,0.720144,0.522798,0.297693,0.577565,0.688738,0.543532,0.302822,50.134236,72.067531,50.675120,204.688975,99.138370,5.414715,15.382393,23.221637,19.733319,5.908666,0.317850,0.271387,0.269076,0.251110,0.366854
3,AR,001,2013,0.329692,0.500504,0.791453,0.793303,0.506448,0.234499,0.391561,0.663588,0.638052,0.320500,148.720574,51.512898,43.286920,53.518767,117.369536,0.000000,6.385274,8.180383,21.909705,11.165014,0.423768,0.377407,0.276310,0.267460,0.283379
4,AR,001,2014,0.344876,0.600713,0.835512,0.789398,0.490574,0.231448,0.461770,0.685360,0.633101,0.302474,143.251276,166.668723,87.333464,91.058577,34.263461,0.000000,0.000000,0.052910,0.752947,0.000000,0.410152,0.401975,0.340921,0.302422,0.331560


## 5. Merge features with USDA outcome variables

This creates the final model-ready county-year dataset.


In [10]:
# ============================================================
# 5. Merge GEE X features with USDA Y outcomes
# ============================================================

merge_keys = ["state_alpha", "county_ansi", "year"]

model_ready = county_outcomes.merge(
    gee_wide,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

print("USDA real county-year rows:", len(county_outcomes))
print("GEE county-year rows:", len(gee_wide))
print("Final model-ready rows:", len(model_ready))

display(model_ready.head())


USDA real county-year rows: 14600
GEE county-year rows: 21312
Final model-ready rows: 14600


,state_alpha,county_ansi,county_name,year,yield_bu_acre,harvested_acres,production_bu,ndvi_05,ndvi_06,ndvi_07,ndvi_08,ndvi_09,evi_05,evi_06,evi_07,evi_08,evi_09,rain_mm_05,rain_mm_06,rain_mm_07,rain_mm_08,rain_mm_09,heat_days_gt35c_05,heat_days_gt35c_06,heat_days_gt35c_07,heat_days_gt35c_08,heat_days_gt35c_09,soil_moisture_l2_05,soil_moisture_l2_06,soil_moisture_l2_07,soil_moisture_l2_08,soil_moisture_l2_09
0,IL,011,BUREAU,2010,55.0,119600,6582000,0.379917,0.702529,0.894240,0.832210,0.356238,0.249877,0.518856,0.768915,0.656171,0.210645,149.427697,221.528352,96.391225,103.732416,58.953503,0.0,0.0,0.0,0.0,0.0,0.356960,0.389571,0.329416,0.326436,0.328387
1,IL,015,CARROLL,2010,58.9,34000,2004000,0.381125,0.703468,0.892417,0.835877,0.427025,0.267056,0.523145,0.798482,0.669357,0.250003,87.404447,198.481283,247.093217,100.095097,46.664447,0.0,0.0,0.0,0.0,0.0,0.360169,0.388912,0.354695,0.353248,0.349173
2,IL,073,HENRY,2010,56.6,151300,8561000,0.372083,0.660423,0.886989,0.839799,0.417788,0.244036,0.488688,0.754704,0.675074,0.248230,146.592249,214.898199,91.043356,157.092745,75.466256,0.0,0.0,0.0,0.0,0.0,0.363602,0.388470,0.328224,0.347568,0.352751
3,IL,085,JO DAVIESS,2010,55.1,32600,1796000,0.451447,0.697857,0.881222,0.813112,0.495007,0.326746,0.518894,0.729695,0.635434,0.293286,121.015765,192.405355,301.160837,114.887953,59.428421,0.0,0.0,0.0,0.0,0.0,0.366842,0.392166,0.363232,0.357552,0.349337
4,IL,103,LEE,2010,56.2,91600,5152000,0.385679,0.728452,0.894631,0.800158,0.336069,0.247326,0.520052,0.781647,0.625257,0.191751,128.915981,187.255665,108.907562,106.267275,42.680307,0.0,0.0,0.0,0.0,0.0,0.348128,0.385681,0.330311,0.324923,0.320376


## 6. Sanity checks

These checks help confirm that the merge did not lose too many rows and that the feature columns are available for modeling.


In [11]:
# ============================================================
# 6. Sanity checks
# ============================================================

monthly_feature_cols = [
    c for c in model_ready.columns
    if any(c.startswith(prefix) for prefix in [
        "ndvi_",
        "evi_",
        "rain_mm_",
        "heat_days_gt35c_",
        "soil_moisture_l2_",
    ])
]

print("Number of monthly feature columns:", len(monthly_feature_cols))
print(monthly_feature_cols)

coverage_by_year = (
    model_ready
    .groupby("year", as_index=False)
    .agg(
        rows=("yield_bu_acre", "size"),
        states=("state_alpha", "nunique"),
        counties=("county_ansi", "count"),
        mean_yield=("yield_bu_acre", "mean"),
        total_harvested_acres=("harvested_acres", "sum"),
    )
)

display(coverage_by_year)

missing_rate = (
    model_ready[monthly_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

display(missing_rate.head(20))


Number of monthly feature columns: 25
['ndvi_05', 'ndvi_06', 'ndvi_07', 'ndvi_08', 'ndvi_09', 'evi_05', 'evi_06', 'evi_07', 'evi_08', 'evi_09', 'rain_mm_05', 'rain_mm_06', 'rain_mm_07', 'rain_mm_08', 'rain_mm_09', 'heat_days_gt35c_05', 'heat_days_gt35c_06', 'heat_days_gt35c_07', 'heat_days_gt35c_08', 'heat_days_gt35c_09', 'soil_moisture_l2_05', 'soil_moisture_l2_06', 'soil_moisture_l2_07', 'soil_moisture_l2_08', 'soil_moisture_l2_09']


,year,rows,states,counties,mean_yield,total_harvested_acres
0,2010,1041,15,1041,43.451969,68452430
1,2011,1011,15,1011,41.414738,65531000
2,2012,1000,15,1000,39.167000,66943930
3,2013,930,15,930,43.983871,64361740
4,2014,959,15,959,47.129718,69947780
5,2015,900,15,900,48.520111,66363910
6,2016,928,15,928,52.682651,69491520
7,2017,942,15,942,49.350106,76843880
8,2018,841,15,841,50.911772,68960480
9,2019,765,15,765,47.184706,52912390


,0
ndvi_05,0.000068
ndvi_06,0.000068
ndvi_07,0.000068
ndvi_08,0.000068
ndvi_09,0.000068
evi_05,0.000068
evi_06,0.000068
evi_07,0.000068
evi_08,0.000068
evi_09,0.000068


## 7. Save model-ready data

The next notebook, `02_yield_prediction_models.ipynb`, should read this output instead of recomputing the whole preparation pipeline.


In [12]:
# ============================================================
# 7. Save model-ready dataset for downstream notebooks
# ============================================================

output_path = PROCESSED_DIR / "model_ready_soybean_15states_2010_2025.csv"

model_ready.to_csv(output_path, index=False)

print("Saved model-ready data to:")
print(output_path)

# Optional: save a small sample for GitHub preview
sample_path = PROCESSED_DIR / "sample_model_ready_soybean.csv"
model_ready.head(100).to_csv(sample_path, index=False)

print("Saved sample data to:")
print(sample_path)


Saved model-ready data to:
/content/gdrive/MyDrive/Soybean_Project_GEE/processed/model_ready_soybean_15states_2010_2025.csv
Saved sample data to:
/content/gdrive/MyDrive/Soybean_Project_GEE/processed/sample_model_ready_soybean.csv


## Notes for GitHub

For the public GitHub repository:

- Do not upload large raw USDA/GEE CSV files.
- Keep this notebook as reproducible code.
- Upload only small sample data or final result tables if needed.
- The full `model_ready_soybean_15states_2010_2025.csv` can remain in Google Drive and be regenerated by running this notebook.
